<a href="https://colab.research.google.com/github/zainkhan-dev/flyrank-ml-internship-week1/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zainkhan-dev/flyrank-ml-internship-week1/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

**The rule, in plain words:** A page is worth a CTR review first if it's already earning real
search visibility at a good position, but its actual click-through rate falls well short of what
other pages sitting at that same position typically get. The bigger that gap, and the more
impressions behind it, the higher it's ranked. This mirrors the product's `needs_ctr_fix` flag —
built here from scratch, honestly, as a baseline to beat, not copied.

### Signal check 1 — staleness (behind FlyRank's refresh flags)

**Claim:** "The longer it's been since a visible page was last updated, the more likely it is to
be declining" — this is the premise behind the product's staleness/refresh flags.

**Test:** bucket `freshness_tier` (from `days_since_last_update`) against the decline rate
(`trend_direction == "down"`), restricted to *visible* pages (`impressions_90d >= 500`, the same
volume floor the session's own `stale_visible_page` reason code used). Table + n printed below.

**Result:** decline rate rises only slightly from the freshest bucket (0-30 days: 58.3%, n=10,063)
to the stalest reliable bucket (91-180 days: 61.6%, n=6,558) — about a 3.3pp difference, in the
expected direction but small next to the 54.2% base rate across all 30,000 rows. The most extreme
bucket (181+ days) shows a striking 94.1% decline rate, but at n=17 it sits well below the ~50-row
floor the auditing-signals skill sets for a trustworthy verdict — that number is noise wearing a
costume, not confirmation.

**Verdict: MIXED.** Direction is right (staler → slightly more likely declining), but the effect
is weak where I actually have enough rows to trust it, and the one bucket that looked dramatic
doesn't have enough data to say anything. This is the "clearly-explained negative" that saves the
rule: I'm **not** letting staleness drive the score below, because on this slice it isn't earning
that trust yet.

### Signal check 2 — CTR vs. position (behind FlyRank's CTR-fix logic)

**Claim:** "Click-through rate should fall as position gets worse" — this is the premise behind
the product's `needs_ctr_fix` flag: a page whose CTR is *below* what its position normally earns
is a real, fixable problem, not just bad luck.

**Test:** for pages with real position data and a volume floor (`avg_position > 0`,
`impressions_90d >= 100`), compute the **weighted** CTR per `position_tier`
(`sum(clicks) / sum(impressions)`, not the mean of per-page rates — averaging raw rates would let
tiny pages dominate). Table + n printed below.

**Result:** weighted CTR falls almost monotonically as position worsens — top_3: 0.487%
(n=533) → page_1: 0.350% (n=8,633) ≈ striking: 0.347% (n=5,903) → page_3_5: 0.155% (n=6,058) →
deep: 0.039% (n=879). Every bucket clears the sample floor by a wide margin. The middle two tiers
(page_1 and striking) are essentially tied, but the overall gradient — best position gets several
times the clicks of the worst — holds clearly.

**Verdict: CONFIRMED.** Position and CTR are genuinely tied in this data, which means a page
sitting well *below* its own tier's typical CTR is a measurable, defensible anomaly — exactly
what the CTR-fix logic assumes. This is the signal my rule below is built on.

**Why the rule leans on signal 2, not signal 1:** signal 2 held up at every sample size I tested;
signal 1 only held up weakly, and its strongest-looking result came from too few rows to trust.
A baseline that's "honestly beatable" should be built on the signal that actually survived the
check — so the score below scores CTR-vs-position, and staleness is left out rather than bolted
on for show.

In [10]:
import pandas as pd
import numpy as np
import os

DATA_PATH = "../../data/raw/content_refresh_anonymized.csv"
if not os.path.exists(DATA_PATH):
    REPO_URL = "https://github.com/zainkhan-dev/flyrank-ml-internship-week1.git"
    if not os.path.exists("flyrank-ml-internship-week1"):
        os.system(f"git clone --quiet {REPO_URL}")
    DATA_PATH = "flyrank-ml-internship-week1/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)
print("rows, cols:", df.shape)

df["is_declining"] = (df["trend_direction"] == "down").astype(int)
visible = df[df["impressions_90d"] >= 500].copy()

tier_order = ["0-30", "31-90", "91-180", "181+"]
signal1_table = (
    visible.groupby("freshness_tier")["is_declining"]
    .agg(decline_rate="mean", n="count")
    .reindex(tier_order)
)
print("\nSignal 1 — staleness vs. decline rate (visible pages, impressions_90d >= 500)")
print(signal1_table)
print(f"\noverall decline rate, all {len(df)} rows: {df['is_declining'].mean():.3f}")

eligible = df[(df["avg_position"] > 0) & (df["impressions_90d"] >= 100)].copy()
tier_order2 = ["top_3", "page_1", "striking", "page_3_5", "deep"]
signal2_table = eligible.groupby("position_tier").agg(
    clicks=("clicks_90d", "sum"),
    impressions=("impressions_90d", "sum"),
    n=("content_id", "count"),
)
signal2_table["weighted_ctr_pct"] = signal2_table["clicks"] / signal2_table["impressions"] * 100
signal2_table = signal2_table.reindex(tier_order2)
print("\nSignal 2 — weighted CTR by position tier (avg_position > 0, impressions_90d >= 100)")
print(signal2_table)

rows, cols: (30000, 44)

Signal 1 — staleness vs. decline rate (visible pages, impressions_90d >= 500)
                decline_rate      n
freshness_tier                     
0-30                0.582530  10063
31-90               0.522727     88
91-180              0.615584   6558
181+                0.941176     17

overall decline rate, all 30000 rows: 0.542

Signal 2 — weighted CTR by position tier (avg_position > 0, impressions_90d >= 100)
               clicks  impressions     n  weighted_ctr_pct
position_tier                                             
top_3           34222      7025180   533          0.487133
page_1         313097     89493618  8633          0.349854
striking        79584     22946217  5903          0.346828
page_3_5        54370     35141017  6058          0.154719
deep              479      1213203   879          0.039482


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

**Score (transparent, no fitted weights):**

1. Compute each `position_tier`'s expected CTR as the weighted CTR from signal check 2 above
   (`sum(clicks) / sum(impressions)` per tier, on the same volume-floored slice).
2. A page is **eligible** to be scored if it has real position data and clears the volume floor
   (`avg_position > 0` and `impressions_90d >= 100`) — below that floor, one click swings CTR too
   much to trust.
3. `ctr_gap_pct = expected_ctr_for_its_tier − its_own_ctr`. Positive means it's underperforming
   its tier.
4. `score = ctr_gap_pct * log1p(impressions_90d)` when the gap is positive and the page is
   eligible, else `0`. The `log1p` on impressions means a big gap on a high-traffic page outranks
   the same gap on a barely-visible one — readable on purpose, same pattern as the session's
   `stale * visible * impressions` rule.

**Reason code (one):** `ctr_below_position_expectation` — attached to every row with `score > 0`.
Rows that don't clear the volume/eligibility bar, or whose CTR already meets or beats their
tier's expectation, carry no reason code and score `0`.

**Action label:** `review_ctr` when `score > 0`, else `monitor`.

**No future-window or label-derived inputs:** the rule only touches `avg_position`, `ctr`,
`clicks_90d`, `impressions_90d`, and `position_tier` — all observable in the current 90-day
window, all pre-decision. `trend_pct`, `trend_direction`, and `is_declining_label` never appear
below.

In [11]:
VOLUME_FLOOR = 100

eligible = df[(df["avg_position"] > 0) & (df["impressions_90d"] >= VOLUME_FLOOR)].copy()
tier_expected_ctr = (
    eligible.groupby("position_tier")
    .apply(lambda g: g["clicks_90d"].sum() / g["impressions_90d"].sum() * 100)
)

df["expected_ctr_pct"] = df["position_tier"].map(tier_expected_ctr)
df["is_eligible"] = ((df["avg_position"] > 0) & (df["impressions_90d"] >= VOLUME_FLOOR)).astype(int)
df["ctr_gap_pct"] = np.where(df["is_eligible"] == 1, df["expected_ctr_pct"] - df["ctr"], np.nan)

df["baseline_action_score"] = np.where(
    (df["is_eligible"] == 1) & (df["ctr_gap_pct"] > 0),
    df["ctr_gap_pct"] * np.log1p(df["impressions_90d"]),
    0.0,
)
df["reason_code"] = np.where(df["baseline_action_score"] > 0, "ctr_below_position_expectation", "none")
df["suggested_action"] = np.where(df["baseline_action_score"] > 0, "review_ctr", "monitor")

ranked = df.sort_values("baseline_action_score", ascending=False).reset_index(drop=True)
ranked["baseline_rank"] = ranked.index + 1

output_columns = [
    "baseline_rank", "content_id", "client_id",
    "baseline_action_score", "reason_code", "suggested_action",
    "position_tier", "avg_position", "impressions_90d", "clicks_90d",
    "ctr", "expected_ctr_pct", "ctr_gap_pct",
    "word_count", "content_type", "days_since_last_update",
]

import os
out_path = "../outputs/baseline_action_score.csv"
if not os.path.isdir(os.path.dirname(out_path)) and os.path.exists("flyrank-ml-internship-week1"):
    out_path = "flyrank-ml-internship-week1/work/outputs/baseline_action_score.csv"
os.makedirs(os.path.dirname(out_path), exist_ok=True)
ranked[output_columns].to_csv(out_path, index=False)

print(f"wrote {out_path} — {len(ranked)} rows")
print(f"rows flagged review_ctr: {(ranked['suggested_action'] == 'review_ctr').sum()}")
print(f"rows flagged monitor: {(ranked['suggested_action'] == 'monitor').sum()}")
print()
print(ranked[output_columns].head(10).to_string(index=False))

/tmp/ipykernel_729/2667597348.py:6: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g["clicks_90d"].sum() / g["impressions_90d"].sum() * 100)


wrote flyrank-ml-internship-week1/work/outputs/baseline_action_score.csv — 30000 rows
rows flagged review_ctr: 15403
rows flagged monitor: 14597

 baseline_rank           content_id         client_id  baseline_action_score                    reason_code suggested_action position_tier  avg_position  impressions_90d  clicks_90d  ctr  expected_ctr_pct  ctr_gap_pct  word_count    content_type  days_since_last_update
             1 content_8451fc6f034d client_d029fa3a95               5.720609 ctr_below_position_expectation       review_ctr         top_3           2.3           272144          75 0.03          0.487133     0.457133      3528.0 keyword article                      20
             2 content_4a6607efcb46 client_6208ef0f77               5.611244 ctr_below_position_expectation       review_ctr         top_3           2.2           128068          17 0.01          0.487133     0.477133      4939.0 keyword article                     104
             3 content_e12868d1f396 client_4

## 3. Top-10 review

*For each of your top ten: one line each — the action, why it's there, and what would make it wrong.*

All ten sit in `position_tier == top_3` (avg position 0.7–2.9), with impressions from ~12k to
~509k — real, high-visibility pages. Every one is `keyword article` / informational-or-adjacent
intent, which is worth noting as a pattern, not a coincidence I engineered.

1. **content_8451fc6f034d** — `review_ctr`. Position 2.3, 272k impressions, CTR 0.03% vs. an
   expected 0.49% for top_3 — a huge gap on huge volume. **Wrong if:** the title/snippet already
   changed after this window closed, or the page is intentionally non-clickable (e.g. it ranks
   for a query it doesn't actually want, like a support doc surfacing for a commercial term).
2. **content_4a6607efcb46** — `review_ctr`. Position 2.2, 128k impressions, CTR 0.01% — almost no
   clicks despite prime placement. **Wrong if:** the SERP for this query is dominated by a
   featured snippet or People-Also-Ask box eating clicks before anyone reaches the list.
3. **content_e12868d1f396** — `review_ctr`. Position 2.9, 150k impressions, CTR 0.07%.
   **Wrong if:** `avg_position` is an average over a volatile 90 days and this page briefly ranked
   much worse for part of the window — the single "2.9" hides that swing.
4. **content_cbdf5a78dcd0** — `review_ctr`. Position 2.4, 15k impressions, CTR 0.02% (3 clicks).
   **Wrong if:** 3 clicks out of 15k is close enough to normal variance at this position tier that
   the "gap" is mostly noise, not a real title/meta problem.
5. **content_d225ec9f3d46** — `review_ctr`. Position 0.7 (essentially rank 1), 26k impressions,
   CTR 0.05%. **Wrong if:** this keyword is branded/navigational and searchers are clicking a
   sitelink or a different result entirely, not skipping this one out of disinterest.
6. **content_8c19996aa890** — `review_ctr`. Position 2.5, 509k impressions (the largest in the
   top 10), CTR 0.15% — still under the 0.49% expectation, but the gap here is smaller than
   rows 1–5; it ranks 6th mostly because of its sheer volume. **Wrong if:** 0.15% is actually
   fine for this specific query's intent (e.g. informational queries with a direct-answer SERP
   feature naturally suppress CTR industry-wide).
7. **content_8053a66bd6ac** — `review_ctr`. Position 2.6, 53k impressions, CTR 0.08%,
   `word_count` missing. **Wrong if:** the missing word count means I can't see whether this is
   thin or well-developed content — the CTR gap may trace back to something my rule can't see.
8. **content_b7bd590fe572** — `review_ctr`. Position 2.9, 12k impressions, CTR 0.02% (3 clicks),
   `word_count` missing. **Wrong if:** same missing-data caveat as #7, plus this is the
   lowest-volume page in the top 10 — closest to the noise floor.
9. **content_998f6f88784c** — `review_ctr`. Position 2.6, 12k impressions, CTR 0.02% (3 clicks).
   **Wrong if:** like #4 and #8, low absolute impressions make a 3-click CTR fragile — a
   confidence interval, not just a point estimate, would separate real gaps from small-sample
   ones.
10. **content_f4e210ee0c27** — `review_ctr`. Position 1.6, 25k impressions, CTR 0.06%.
    **Wrong if:** the page recently changed title/meta and this 90-day window still reflects the
    *old* version's performance — the fix may already be live and this row is stale evidence.

In [12]:
display_cols = [
    "baseline_rank", "content_id", "position_tier", "avg_position",
    "impressions_90d", "clicks_90d", "ctr", "expected_ctr_pct", "ctr_gap_pct",
    "word_count", "content_type", "suggested_action", "reason_code",
]
top10 = ranked[display_cols].head(10)
print(top10.to_string(index=False))

 baseline_rank           content_id position_tier  avg_position  impressions_90d  clicks_90d  ctr  expected_ctr_pct  ctr_gap_pct  word_count    content_type suggested_action                    reason_code
             1 content_8451fc6f034d         top_3           2.3           272144          75 0.03          0.487133     0.457133      3528.0 keyword article       review_ctr ctr_below_position_expectation
             2 content_4a6607efcb46         top_3           2.2           128068          17 0.01          0.487133     0.477133      4939.0 keyword article       review_ctr ctr_below_position_expectation
             3 content_e12868d1f396         top_3           2.9           149712         104 0.07          0.487133     0.417133      2363.0 keyword article       review_ctr ctr_below_position_expectation
             4 content_cbdf5a78dcd0         top_3           2.4            14830           3 0.02          0.487133     0.467133      3187.0 keyword article       review_ctr ctr_be

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

**Weak picks — three real ones, further down the queue, not the top 10 above:**

Rows #7, #8, and #9 in the top 10 already flag one weakness by themselves: two have missing
`word_count`, and three (#4, #8, #9) are built on only 3 clicks. But the clearer weak-pick case
sits further down the ranked list, printed below — pages sitting right at the `impressions_90d
>= 100` eligibility floor with **zero clicks**, spread across every position tier (ranks roughly
6,000-15,000, well below the top 10). At exactly 100-101 impressions, zero clicks isn't unusual
under normal variance for a sub-0.5% CTR; the rule can't yet tell "genuinely broken title" from
"small sample, nothing happened to click on yet." These score far lower than the true top 10 —
the `log1p(impressions)` term is doing its job — but they're a reminder that the floor of 100
impressions is a minimum for eligibility, not a guarantee of a trustworthy signal. A future
version should widen the volume floor or add a confidence interval on `ctr_gap_pct` instead of
a bare threshold.

**Leakage check:** the rule and score use exactly five columns — `avg_position`, `ctr`,
`clicks_90d`, `impressions_90d`, `position_tier` — all observable in the current 90-day window,
all pre-decision. `trend_pct` and `trend_direction` (the label source) are never referenced in
the scoring cell, and no product decision flags (`health_score`, `needs_ctr_fix`,
`priority_score`, `action_type`) exist in this dataset to leak in the first place — confirmed
against the column list below.

In [13]:
weak_picks = ranked[
    (ranked["suggested_action"] == "review_ctr") & (ranked["clicks_90d"] == 0)
].sort_values("impressions_90d").head(10)

print("Weak picks — zero clicks, near the volume floor:")
print(weak_picks[["baseline_rank", "content_id", "position_tier", "impressions_90d",
                   "clicks_90d", "ctr", "baseline_action_score"]].to_string(index=False))

scoring_inputs = {"avg_position", "ctr", "clicks_90d", "impressions_90d", "position_tier"}
forbidden = {"trend_pct", "trend_direction", "is_declining_label",
             "health_score", "needs_ctr_fix", "priority_score", "action_type"}

print("\nScoring inputs used:", sorted(scoring_inputs))
print("Forbidden columns present anywhere in dataset columns:",
      sorted(forbidden & set(df.columns)))
print("Forbidden columns used to build the score:", sorted(forbidden & scoring_inputs))
assert not (forbidden & scoring_inputs), "leakage: a forbidden column feeds the score"
print("\nNo leakage: scoring inputs and forbidden/label columns do not overlap.")

Weak picks — zero clicks, near the volume floor:
 baseline_rank           content_id position_tier  impressions_90d  clicks_90d  ctr  baseline_action_score
         14873 content_1712d0a66030          deep              100           0  0.0               0.182215
          6366 content_f31afa6a8974      striking              100           0  0.0               1.600655
          6367 content_ce43d8d8f2e4      striking              100           0  0.0               1.600655
          6368 content_d0debd0920f8      striking              100           0  0.0               1.600655
          6365 content_476b20d68337      striking              100           0  0.0               1.600655
          6364 content_3d5c5db1d307      striking              100           0  0.0               1.600655
          6271 content_62941b746237        page_1              100           0  0.0               1.614618
         12110 content_fb0e174a7284      page_3_5              100           0  0.0            

## Self-check

Before you submit, confirm each line honestly:

- [✅ True] Every section above is filled — markdown thinking AND the code that backs it
- [✅ True] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✅ True] No client names, URLs, or private queries anywhere
- [✅ True] My claims use careful words: observed, measured, directional, decision-support
- [✅ True] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.